# Black-Litterman Research Lab v15 — Readable Monthly Investing Comparison

Bu notebook aylık yatırım yapan biri için portföy kurma yöntemlerini aynı veri üzerinde karşılaştırır.

Stratejiler:
- **Equal**
- **HRP**
- **BL**
- **HYBRID**

Amaç:
- hangi strateji daha çok büyüyor görmek
- bunu yaparken ne kadar düştüğünü görmek
- gerçekten çeşitleniyor mu anlamak
- akıllı yöntemler basit benchmark'ları geçiyor mu görmek


## Bu notebook nasıl okunmalı?

Şu sırayla oku:

1. **Summary table**
2. **Winner board**
3. **Portföy değeri vs katkılar**
4. **Drawdown**
5. **Benchmark comparison**
6. **Latest weights + sleeve summary**
7. **Concentration ve turnover**
8. **Heatmap ve BL diagnostics**


In [1]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 5.1 MB/s eta 0:00:00


In [2]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Configuration

Bu bölüm yatırım evrenini, backtest ayarlarını ve strateji ayarlarını tanımlar.

Not:
- Aşağıdaki BIST listesi canlı doğrulanmış resmi güncel BIST100 listesi gibi ele alınmamalı.
- Bu, önceki denemelerden gelen başlangıç evrenidir.


In [3]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

FX_TICKERS = {"USDTRY": "USDTRY=X", "EURTRY": "EURTRY=X"}
METAL_USD_TICKERS = {"ALTIN_USD": "GC=F", "GUMUS_USD": "SI=F", "PLATIN_USD": "PL=F"}

BACKTEST = {
    "download_period": "10y",
    "min_history_days": 180,
    "lookback_days": 252,
    "test_months": 6,
    "monthly_contribution_try": 25000,
}

STRATEGY_CFG = {
    "strategies": ["Equal", "HRP", "BL", "HYBRID"],
    "max_weight": 0.10,
    "hybrid_alpha": 0.25,
    "gold_label": "ALTIN_TRY",
    "gold_min": 0.05,
    "gold_max": 0.20,
}

print("BIST başlangıç evreni:", len(BIST_TICKERS))
print("Stratejiler:", STRATEGY_CFG["strategies"])


BIST başlangıç evreni: 71
Stratejiler: ['Equal', 'HRP', 'BL', 'HYBRID']


## 2) Helper functions

Bu bölüm veri indirme, temizlik, ağırlık normalizasyonu ve strateji fonksiyonlarını içerir.


In [4]:
def try_fmt(x):
    if pd.isna(x):
        return "-"
    return f"₺{x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct_fmt(x):
    if pd.isna(x):
        return "-"
    return f"{x*100:.2f}%"

def safe_asset_name(symbol):
    return symbol.replace(".IS", "").replace("=X", "").replace("=F", "")

def normalize_prices(df):
    return df / df.iloc[0] * 100

def fetch_close_series(symbol, period):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs("Close", axis=1, level=0) if "Close" in level0 else df.iloc[:, :1]
    else:
        close = df[["Close"]] if "Close" in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def clean_returns(df):
    rets = df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="any")
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1 / len(index), index=index)
    return w / s

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return normalize_weights(w, w.index)
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return normalize_weights(w, w.index)
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return normalize_weights(w, w.index)

def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print("HRP failed, inverse-vol kullanıldı:", str(e))
        return inverse_vol_weight(price_window)

def build_view_table(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return pd.DataFrame({"asset": price_window.columns, "ann_return": 0.0, "ann_vol": np.nan, "raw_score": 0.0, "view": 0.0})
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    view = score.clip(-0.15, 0.15)
    out = pd.DataFrame({"asset": score.index, "ann_return": ann.reindex(score.index).values, "ann_vol": vol.reindex(score.index).values, "raw_score": score.values, "view": view.values})
    return out

def views_from_prices(price_window):
    vt = build_view_table(price_window)
    return dict(zip(vt["asset"], vt["view"]))

def bl_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({"asset": list(price_window.columns), "view": [0.0]*len(price_window.columns), "view_source": "rules_insufficient_data"})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views = views_from_prices(price_window)
        bl = BlackLittermanModel(S, pi="equal", absolute_views=views, omega="default")
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, STRATEGY_CFG["max_weight"]))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, STRATEGY_CFG["gold_label"], STRATEGY_CFG["gold_min"], STRATEGY_CFG["gold_max"])
        diag = pd.DataFrame({"asset": list(views.keys()), "view": list(views.values()), "view_source": "rules"})
        return w, diag
    except Exception as e:
        print("BL failed, inverse-vol kullanıldı:", str(e))
        w = inverse_vol_weight(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({"asset": list(fallback_views.keys()), "view": list(fallback_views.values()), "view_source": "rules_bl_fallback"})
        return w, diag

def hybrid_weights(price_window):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window)
    a = STRATEGY_CFG["hybrid_alpha"]
    w = (1 - a) * w_hrp + a * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=STRATEGY_CFG["max_weight"])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, STRATEGY_CFG["gold_label"], STRATEGY_CFG["gold_min"], STRATEGY_CFG["gold_max"])
    return w, diag

def compute_weights(strategy_name, price_window):
    if strategy_name == "Equal":
        w = equal_weight(price_window.columns)
        w = w.clip(upper=STRATEGY_CFG["max_weight"])
        w = normalize_weights(w, price_window.columns)
        return w, None
    if strategy_name == "HRP":
        w = hrp_weights_safe(price_window)
        w = w.clip(upper=STRATEGY_CFG["max_weight"])
        w = normalize_weights(w, price_window.columns)
        return w, None
    if strategy_name == "BL":
        return bl_weights_safe(price_window)
    if strategy_name == "HYBRID":
        return hybrid_weights(price_window)
    raise ValueError(f"Unknown strategy: {strategy_name}")

def get_rebalance_dates(prices, months):
    monthly = prices.resample("MS").first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))


## 3) Data loading and coverage

Bu bölüm en kritik güven kontrolüdür.

Bakman gereken şeyler:
- veri gerçekten inmiş mi?
- hangi varlıklar testte kalmış?
- ortak tarihçe yeterli mi?


In [5]:
rows = []
series_list = []

for sym in BIST_TICKERS:
    s = fetch_close_series(sym, BACKTEST["download_period"])
    rows.append({"source": sym, "asset": safe_asset_name(sym), "group": "BIST", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})
    if s is not None and len(s) >= BACKTEST["min_history_days"]:
        s = s.copy()
        s.name = safe_asset_name(sym)
        series_list.append(s)

fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym, BACKTEST["download_period"])
    fx_series[label] = s
    rows.append({"source": sym, "asset": label, "group": "FX", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})
    if s is not None and len(s) >= BACKTEST["min_history_days"]:
        s = s.copy()
        s.name = label
        series_list.append(s)

metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym, BACKTEST["download_period"])
    metal_usd[label] = s
    rows.append({"source": sym, "asset": label, "group": "METAL_USD", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})

usdtry = fx_series.get("USDTRY")
for try_label, usd_label in [("ALTIN_TRY", "ALTIN_USD"), ("GUMUS_TRY", "GUMUS_USD"), ("PLATIN_TRY", "PLATIN_USD")]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({"source": f"{usd_label} * USDTRY", "asset": try_label, "group": "METAL_TRY", "n": len(s), "start": str(s.index.min().date()), "end": str(s.index.max().date()), "status": "ok"})
        if len(s) >= BACKTEST["min_history_days"]:
            series_list.append(s)

coverage = pd.DataFrame(rows)
display(coverage.head())

prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()

asset_group_map = {}
for _, r in coverage.iterrows():
    asset_group_map[r["asset"]] = r["group"]

eligible_assets = pd.DataFrame({"asset": prices.columns, "group": [asset_group_map.get(a, "OTHER") for a in prices.columns]})
monthly_points = prices.resample("MS").first().dropna()
lookback = min(BACKTEST["lookback_days"], max(60, len(prices) - 40))
months = BACKTEST["test_months"]

print("Ortak başlangıç:", common_start)
print("Kullanılan varlık sayısı:", prices.shape[1])
print("Toplam gün:", len(prices), "lookback:", lookback, "test_months:", months)
display(prices.tail())


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: KOZAA.IS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KOZAA.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: KOZAL.IS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KOZAL.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")')


,source,asset,group,n,start,end,status
0,AEFES.IS,AEFES,BIST,2543,2016-03-30,2026-03-30,ok
1,AGHOL.IS,AGHOL,BIST,2543,2016-03-30,2026-03-30,ok
2,AKBNK.IS,AKBNK,BIST,2542,2016-03-30,2026-03-30,ok
3,AKFYE.IS,AKFYE,BIST,761,2023-03-16,2026-03-30,ok
4,AKSA.IS,AKSA,BIST,2543,2016-03-30,2026-03-30,ok


Ortak başlangıç: 2023-09-21 00:00:00
Kullanılan varlık sayısı: 74
Toplam gün: 656 lookback: 252 test_months: 6


,AEFES,AGHOL,AKBNK,AKFYE,AKSA,AKSEN,ALARK,ARCLK,ASELS,ASTOR,BIMAS,BOBET,CCOLA,CIMSA,CWENE,DOAS,DOHOL,ECILC,EGEEN,EKGYO,ENERY,ENJSA,ENKAI,EREGL,EUPWR,FROTO,GARAN,GESAN,GUBRF,HALKB,HEKTS,ISCTR,ISMEN,KAYSE,KCAER,KCHOL,KLSER,KRDMD,MAVI,MGROS,MIATK,ODAS,OTKAR,OYAKC,PETKM,PGSUS,REEDR,SASA,SAHOL,SDTTR,SISE,SKBNK,SMRTG,SOKM,TAVHL,TCELL,THYAO,TKFEN,TOASO,TSKB,TTKOM,TTRAK,TUPRS,ULKER,VAKBN,VESBE,VESTL,YKBNK,ZOREN,USDTRY,EURTRY,ALTIN_TRY,GUMUS_TRY,PLATIN_TRY
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-24,17.320000,27.719999,68.116837,21.580000,10.97,74.800003,89.099998,114.000000,352.00,191.800003,679.5,19.170000,71.750000,48.959999,28.860001,193.500000,20.799999,115.199997,5692.5,19.700001,8.79,114.500000,93.699997,27.780001,35.000000,104.199997,130.000000,43.700001,476.50,38.060001,2.91,13.29,44.779999,4.44,11.40,182.669998,25.900000,29.280001,42.459999,586.0,38.540001,6.09,383.50,24.000000,19.549999,179.000000,7.93,2.38,89.949997,219.000000,47.439999,10.19,7.36,50.599998,291.50,106.199997,291.00,88.250000,276.25,11.524907,57.299999,453.00,259.000000,110.800003,32.279999,7.10,29.059999,33.340000,2.86,44.338600,51.498371,195058.795018,3071.512279,83888.631500
2026-03-25,17.639999,27.660000,68.698196,21.459999,10.81,76.500000,90.000000,114.500000,337.50,209.199997,705.0,19.280001,69.000000,50.450001,29.240000,195.600006,21.139999,115.900002,5715.0,20.400000,8.67,118.199997,93.949997,27.920000,37.680000,104.599998,130.399994,45.099998,471.75,37.820000,2.86,13.42,44.360001,4.36,11.27,186.699997,25.860001,29.440001,42.560001,604.5,37.700001,6.10,379.00,23.900000,19.030001,180.000000,7.69,2.36,91.150002,214.300003,47.160000,10.32,7.30,51.000000,305.00,107.400002,294.50,88.750000,278.50,11.496286,58.799999,451.25,250.500000,111.300003,32.520000,7.06,28.719999,34.040001,2.84,44.339298,51.500900,201734.930510,3208.435963,85362.014823
2026-03-26,17.190001,27.420000,67.250000,19.990000,11.25,74.699997,87.849998,111.300003,338.00,203.199997,673.0,19.059999,70.699997,49.840000,28.520000,193.199997,20.660000,112.000000,5660.0,19.809999,8.55,115.199997,91.849998,26.900000,37.720001,102.699997,126.699997,44.520000,473.50,36.660000,2.84,13.18,42.880001,4.30,10.99,189.300003,25.500000,28.920000,42.680000,600.5,36.919998,5.80,368.00,23.719999,19.170000,177.600006,7.46,2.39,89.949997,214.699997,45.320000,10.14,7.08,48.000000,294.50,106.000000,293.50,93.349998,272.50,11.124207,57.950001,453.50,240.199997,110.599998,31.299999,7.05,28.000000,33.180000,2.78,44.354599,51.379120,194073.547920,3001.519963,81537.061506
2026-03-27,16.730000,27.200001,66.900002,19.420000,10.79,71.599998,86.949997,110.500000,330.75,205.899994,670.5,19.030001,68.349998,49.099998,29.600000,192.899994,21.180000,109.000000,5620.0,19.270000,8.47,117.800003,92.449997,27.900000,37.599998,103.800003,126.500000,44.040001,462.75,36.799999,2.80,13.18,43.200001,4.26,11.00,192.600006,25.299999,29.400000,42.139999,603.0,37.599998,5.68,364.25,23.620001,19.200001,177.000000,7.14,2.33,89.099998,207.500000,44.880001,10.13,7.01,48.180000,292.75,106.400002,294.00,97.699997,279.75,11.100000,58.500000,450.00,240.500000,111.500000,31.040001,6.99,28.440001,33.180000,2.74,44.455551,51.317211,199694.335754,3091.661223,83158.552891
2026-03-30,16.170000,26.959999,65.599998,21.360001,10.57,78.750000,87.750000,108.000000,320.50,195.199997,677.0,18.920000,65.650002,48.599998,29.600000,189.500000,20.379999,107.000000,5517.5,19.170000,8.58,116.199997,93.449997,27.580000,40.500000,100.000000,125.500000,45.000000,465.00,35.779999,2.80,12.94,42.639999,4.34,10.80,194.000000,25.059999,29.120001,41.000000,594.5,41.360001,5.77,368.25,23.459999,20.500000,174.600006,7.01,2.33,88.900002,207.800003,43.099998,10.40,6.90,49.040001,292.50,104.599998,288.75,96.300003,271.50,11.080000,58.500000,444.00,247.000000,112.500000,30.100000,6.96,28.620001,32.720001,2.77,44.463402,50.951431,203202.196883,3152.232721,84836.170624


### Coverage charts nasıl okunur?

İlk grafik:
- her grupta kaç seri indirilebildiğini gösterir

İkinci grafik:
- indirilenler içinden kaçının gerçekten testte kaldığını gösterir


In [6]:
coverage_counts = coverage.groupby(["group", "status"]).size().reset_index(name="count")
fig = px.bar(coverage_counts, x="group", y="count", color="status", barmode="group", title="Veri kapsamı: grup bazında indirilen seriler")
fig.show()

eligible_counts = eligible_assets.groupby("group").size().reset_index(name="eligible_count")
fig = px.bar(eligible_counts, x="group", y="eligible_count", title="Teste kalan varlık sayısı")
fig.show()


## 4) Market context

Burada stratejiyi değil, piyasa ortamını görüyoruz.

Bakman gereken şeyler:
- BIST basket güçlü müydü?
- FX mi baskındı?
- altın / gümüş / platin nasıl davrandı?


In [7]:
bist_cols = [c for c in prices.columns if c not in ["USDTRY", "EURTRY", "ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]]
bist_basket = (1 + returns[bist_cols].mean(axis=1)).cumprod() if len(bist_cols) else pd.Series(dtype=float)
if len(bist_basket):
    bist_basket.name = "BIST_BASKET"

bench = pd.concat([bist_basket, prices[["USDTRY", "EURTRY", "ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]]], axis=1).dropna()
bench_norm = normalize_prices(bench.tail(504))
fig = px.line(bench_norm, x=bench_norm.index, y=bench_norm.columns, title="Son ~2 yıl benchmark görünümü (normalize)")
fig.show()

bench_rets = bench.pct_change().dropna()
fig = px.imshow(bench_rets.corr(), text_auto=True, aspect="auto", zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Benchmark korelasyon ısı haritası")
fig.show()


## 5) Strategy definitions

### Equal
En sade baseline.

### HRP
Korelasyon yapısını kullanarak riski daha dengeli dağıtmaya çalışır.

### BL
View ekleyerek beklenen getirileri tilt etmeye çalışır.

### HYBRID
HRP ve BL arasında denge denemesi.


## 6) Black-Litterman views

Bu tablo BL tarafının ne düşündüğünü görünür yapar.
- çok fazla tavana vuran view varsa BL kaba olabilir
- daha yumuşak ve farklılaşmış view'lar daha anlamlıdır


In [8]:
latest_view_table = build_view_table(prices.tail(lookback))
display(latest_view_table.sort_values("view", ascending=False).head(15))


,asset,ann_return,ann_vol,raw_score,view
0,AEFES,0.062766,0.377256,0.166375,0.15
2,AKBNK,0.392305,0.390564,1.004458,0.15
9,ASTOR,0.730831,0.436592,1.673944,0.15
3,AKFYE,0.234564,0.354735,0.661238,0.15
4,AKSA,0.078761,0.333433,0.236213,0.15
5,AKSEN,0.943291,0.414816,2.273999,0.15
8,ASELS,1.060614,0.442264,2.398145,0.15
12,CCOLA,0.302467,0.339238,0.891605,0.15
11,BOBET,0.101033,0.378014,0.267273,0.15
10,BIMAS,0.467023,0.330719,1.412147,0.15


## 7) Monthly walk-forward backtest

Bu bölüm gerçek hayata daha yakındır:
- her ay katkı eklenir
- sadece o tarihe kadar olan veri kullanılır
- yeni portföy kurulur
- bir sonraki rebalance tarihine kadar tutulur


In [9]:
def run_strategy(strategy_name, prices, lookback, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []

    for i, dt in enumerate(rebalance_dates):
        cash += BACKTEST["monthly_contribution_try"]
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0

        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)

        for t, v in vals.items():
            hist.append({"date": t, "strategy": strategy_name, "portfolio_value": float(v)})

        row = {"date": dt, "strategy": strategy_name, "capital_after_contribution": total}
        for c, x in weights.items():
            row[f"weight_{c}"] = float(x)
        rebs.append(row)

        if diag is not None:
            tmp = diag.copy()
            tmp["date"] = dt
            tmp["strategy"] = strategy_name
            diags.append(tmp)

    hist_df = pd.DataFrame(hist).drop_duplicates(["date", "strategy"])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

hist_parts, weight_parts, diag_parts = [], [], []
for s in STRATEGY_CFG["strategies"]:
    h, w, d = run_strategy(s, prices, lookback, months)
    hist_parts.append(h)
    weight_parts.append(w)
    if not d.empty:
        diag_parts.append(d)

hist_df = pd.concat(hist_parts, ignore_index=True)
weights_df = pd.concat(weight_parts, ignore_index=True)
diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()

display(hist_df.tail())
display(weights_df.tail())


,date,strategy,portfolio_value
507,2026-03-24,HYBRID,164128.005180
508,2026-03-25,HYBRID,164711.415253
509,2026-03-26,HYBRID,162301.012194
510,2026-03-27,HYBRID,162719.844851
511,2026-03-30,HYBRID,162763.704364


,date,strategy,capital_after_contribution,weight_AEFES,weight_AGHOL,weight_AKBNK,weight_AKFYE,weight_AKSA,weight_AKSEN,weight_ALARK,weight_ARCLK,weight_ASELS,weight_ASTOR,weight_BIMAS,weight_BOBET,weight_CCOLA,weight_CIMSA,weight_CWENE,weight_DOAS,weight_DOHOL,weight_ECILC,weight_EGEEN,weight_EKGYO,weight_ENERY,weight_ENJSA,weight_ENKAI,weight_EREGL,weight_EUPWR,weight_FROTO,weight_GARAN,weight_GESAN,weight_GUBRF,weight_HALKB,weight_HEKTS,weight_ISCTR,weight_ISMEN,weight_KAYSE,weight_KCAER,weight_KCHOL,weight_KLSER,weight_KRDMD,weight_MAVI,weight_MGROS,weight_MIATK,weight_ODAS,weight_OTKAR,weight_OYAKC,weight_PETKM,weight_PGSUS,weight_REEDR,weight_SASA,weight_SAHOL,weight_SDTTR,weight_SISE,weight_SKBNK,weight_SMRTG,weight_SOKM,weight_TAVHL,weight_TCELL,weight_THYAO,weight_TKFEN,weight_TOASO,weight_TSKB,weight_TTKOM,weight_TTRAK,weight_TUPRS,weight_ULKER,weight_VAKBN,weight_VESBE,weight_VESTL,weight_YKBNK,weight_ZOREN,weight_USDTRY,weight_EURTRY,weight_ALTIN_TRY,weight_GUMUS_TRY,weight_PLATIN_TRY
19,2025-11-03,HYBRID,50242.471725,0.001665,0.002123,0.013293,0.003223,0.009171,0.008000,0.002784,0.003463,0.034615,0.002761,0.006299,0.003151,0.002568,0.015546,0.002405,0.004121,0.004289,0.005878,0.003964,0.013096,0.023469,0.005591,0.003636,0.004004,0.002550,0.001931,0.000969,0.003213,0.003509,0.002581,0.002328,0.000965,0.001569,0.005177,0.003133,0.002134,0.002899,0.003025,0.006599,0.040072,0.002589,0.002062,0.003255,0.011611,0.003061,0.003080,0.002172,0.001941,0.001925,0.003853,0.004688,0.003489,0.002971,0.006949,0.023600,0.047150,0.003461,0.009754,0.002165,0.001779,0.002350,0.006736,0.012854,0.001637,0.003608,0.004163,0.003316,0.002586,0.002450,0.176136,0.176136,0.079465,0.067081,0.058158
20,2025-12-01,HYBRID,76891.275274,0.001775,0.002200,0.021733,0.003627,0.003993,0.003604,0.001897,0.002912,0.035663,0.002432,0.001747,0.003400,0.003912,0.003218,0.002532,0.003233,0.028337,0.003445,0.003770,0.002255,0.003592,0.007868,0.002727,0.002706,0.002075,0.018140,0.004712,0.002219,0.003198,0.002058,0.002494,0.004320,0.001852,0.004838,0.002735,0.001785,0.005234,0.001292,0.004308,0.039295,0.002313,0.002237,0.002151,0.010373,0.002151,0.002711,0.002954,0.002013,0.001732,0.003646,0.004107,0.003590,0.003048,0.003084,0.020472,0.041700,0.003277,0.004688,0.002093,0.004561,0.002888,0.006359,0.029214,0.003339,0.006833,0.002571,0.002041,0.002034,0.002585,0.180239,0.180239,0.080744,0.066339,0.058544
21,2026-01-02,HYBRID,106330.691116,0.002769,0.003450,0.009468,0.002660,0.004255,0.003290,0.002254,0.002695,0.022307,0.002465,0.002089,0.003172,0.026896,0.003108,0.002245,0.003712,0.041168,0.002843,0.003058,0.001905,0.006956,0.008474,0.004151,0.002373,0.001929,0.023445,0.004711,0.002281,0.003321,0.001906,0.002541,0.004155,0.001776,0.005107,0.003037,0.001598,0.004948,0.001745,0.024262,0.021878,0.002247,0.002187,0.002123,0.003527,0.002132,0.002295,0.002979,0.001270,0.001619,0.003497,0.001671,0.003197,0.002823,0.002229,0.023315,0.035072,0.003016,0.003139,0.002267,0.004303,0.002834,0.006116,0.038772,0.003335,0.011242,0.002603,0.002194,0.001961,0.002435,0.187415,0.187415,0.076562,0.056150,0.039654
22,2026-02-02,HYBRID,141281.297147,0.001877,0.015583,0.011515,0.002316,0.002806,0.021016,0.001885,0.002439,0.022820,0.001706,0.001160,0.004357,0.004305,0.002057,0.003234,0.002710,0.037999,0.004935,0.003733,0.000864,0.020990,0.019634,0.007809,0.001447,0.001425,0.002732,0.000829,0.002012,0.003626,0.001345,0.001823,0.001644,0.001680,0.008257,0.001598,0.001936,0.003583,0.001076,0.020111,0.032663,0.002922,0.001966,0.002566,0.002472,0.002946,0.002753,0.005606,0.001273,0.001637,0.008872,0.002165,0.003496,0.001880,0.002149,0.033151,0.041630,0.002817,0.017046,0.001890,0.000814,0.001841,0.004604,0.035315,0.001353,0.001840,0.002092,0.001749,0.002391,0.002703,0.197564,0.197564,0.071805,0.027508,0.026082
23,2026-03-02,HYBRID,167567.331468,0.001567,0.001983,0.010612,0.003624,0.001899,0.011535,0.002660,0.002492,0.023181,0.001660,0.030026,0.004039,0.010425,0.002080,0.003276,0.002877,0.050097

## 8) Summary table

Bu tablo en üst seviye sonucu verir.
Tek bir metriğe bakma. Getiri, drawdown, turnover ve konsantrasyon birlikte okunmalı.


In [10]:
summary_rows = []
contributed = BACKTEST["monthly_contribution_try"] * months

for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    monthly_ret = sub.set_index("date")["portfolio_value"].resample("M").last().pct_change().dropna()

    summary_rows.append({
        "Strategy": s,
        "Contributed": try_fmt(contributed),
        "Ending Value": try_fmt(sub["portfolio_value"].iloc[-1]),
        "Net Profit": try_fmt(sub["portfolio_value"].iloc[-1] - contributed),
        "TWR_like": pct_fmt(sub["portfolio_value"].iloc[-1] / contributed - 1),
        "Ann Vol": pct_fmt(sub["ret"].std() * np.sqrt(252)),
        "Sharpe": f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
        "MaxDD": pct_fmt(dd.min()),
        "Avg Monthly": pct_fmt(monthly_ret.mean() if len(monthly_ret) else np.nan),
    })

summary = pd.DataFrame(summary_rows)
display(summary)


,Strategy,Contributed,Ending Value,Net Profit,TWR_like,Ann Vol,Sharpe,MaxDD,Avg Monthly
0,Equal,₺150.000,₺154.413,₺4.413,2.94%,165.97%,2.74,-9.12%,47.32%
1,HRP,₺150.000,₺157.862,₺7.862,5.24%,163.93%,2.78,-3.21%,47.36%
2,BL,₺150.000,₺168.857,₺18.857,12.57%,162.03%,2.89,-7.83%,49.82%
3,HYBRID,₺150.000,₺162.764,₺12.764,8.51%,162.97%,2.83,-5.33%,48.46%


## 9) Winner board

Bu tablo hızlı karar desteği içindir.
Tek bir metriğe bakıp yanlış karar vermemek için özetler.


In [11]:
summary_numeric = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    summary_numeric.append({
        "strategy": s,
        "ending_value": sub["portfolio_value"].iloc[-1],
        "maxdd": dd.min(),
        "sharpe": (sub["ret"].mean() / (sub["ret"].std() + 1e-9) * np.sqrt(252)),
    })
summary_numeric = pd.DataFrame(summary_numeric)

winner_board = pd.DataFrame([
    {"category": "Highest Final Value", "winner": summary_numeric.loc[summary_numeric["ending_value"].idxmax(), "strategy"], "value": try_fmt(summary_numeric["ending_value"].max())},
    {"category": "Lowest Max Drawdown", "winner": summary_numeric.loc[summary_numeric["maxdd"].idxmax(), "strategy"], "value": pct_fmt(summary_numeric["maxdd"].max())},
    {"category": "Best Sharpe", "winner": summary_numeric.loc[summary_numeric["sharpe"].idxmax(), "strategy"], "value": f"{summary_numeric['sharpe'].max():.2f}"},
])
display(winner_board)


,category,winner,value
0,Highest Final Value,BL,₺168.857
1,Lowest Max Drawdown,HRP,-3.21%
2,Best Sharpe,BL,2.89


## 10) Portfolio value vs contributions

Bu grafik aylık yatırım yapan biri için en sezgisel grafiklerden biridir.
Katkı çizgisi ile strateji değer çizgilerini aynı anda görürsün.


In [12]:
rebal_dates = get_rebalance_dates(prices, months)
contrib_series = pd.Series(
    np.arange(1, len(rebal_dates) + 1) * BACKTEST["monthly_contribution_try"],
    index=rebal_dates
).reindex(hist_df["date"].sort_values().unique(), method="ffill")

fig = go.Figure()
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date")
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["portfolio_value"], mode="lines", name=s))
fig.add_trace(go.Scatter(x=contrib_series.index, y=contrib_series.values, mode="lines", name="Contributed Cash", line=dict(dash="dash")))
fig.update_layout(title="Portföy değeri vs kümülatif katkı", template="plotly_white")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
fig.show()


## 11) Performance comparison charts

Bu bölüm üç soruya cevap verir:
1. Hangi strateji daha çok büyümüş?
2. Bunu yaparken ne kadar acı çekmiş?
3. Ay bazında tutarlı mıymış?


In [13]:
fig = px.line(hist_df, x="date", y="portfolio_value", color="strategy", title="Strateji bazında portföy değeri")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
fig.show()

dd_parts = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["drawdown"] = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    dd_parts.append(sub[["date", "strategy", "drawdown"]])
dd_df = pd.concat(dd_parts, ignore_index=True)
fig = px.line(dd_df, x="date", y="drawdown", color="strategy", title="Strateji bazında drawdown")
fig.show()

mr_parts = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy().set_index("date")
    monthly = sub["portfolio_value"].resample("M").last().pct_change().dropna()
    mr_parts.append(pd.DataFrame({"date": monthly.index, "strategy": s, "monthly_return": monthly.values}))
mr_df = pd.concat(mr_parts, ignore_index=True)
fig = px.bar(mr_df, x="date", y="monthly_return", color="strategy", barmode="group", title="Aylık portföy getirileri")
fig.show()


## 12) Benchmark comparison

Akıllı stratejiler sadece birbirini değil, basit benchmark'ları da geçebilmeli.
Burada basit benchmark'larla aynı katkı akışında karşılaştırıyoruz.


In [14]:
def run_fixed_mix(name, prices, alloc_map, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist = []

    for i, dt in enumerate(rebalance_dates):
        cash += BACKTEST["monthly_contribution_try"]
        w = pd.Series(0.0, index=prices.columns)

        for asset, wt in alloc_map.items():
            if asset == "BIST_BASKET":
                bist_assets = [c for c in prices.columns if c in bist_cols]
                if len(bist_assets):
                    w.loc[bist_assets] += wt / len(bist_assets)
            elif asset in prices.columns:
                w.loc[asset] += wt

        w = normalize_weights(w, prices.columns)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (w * total) / current
        cash = 0.0

        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)

        for t, v in vals.items():
            hist.append({"date": t, "strategy": name, "portfolio_value": float(v)})
    return pd.DataFrame(hist).drop_duplicates(["date", "strategy"])

benchmark_defs = {
    "BIST_BASKET": {"BIST_BASKET": 1.0},
    "USD_ONLY": {"USDTRY": 1.0},
    "GOLD_ONLY": {"ALTIN_TRY": 1.0},
    "USD_GOLD_50_50": {"USDTRY": 0.5, "ALTIN_TRY": 0.5},
    "SIMPLE_60_20_20": {"BIST_BASKET": 0.6, "USDTRY": 0.2, "ALTIN_TRY": 0.2},
}

bench_hist_parts = []
for n, alloc in benchmark_defs.items():
    bench_hist_parts.append(run_fixed_mix(n, prices, alloc, months))
bench_hist_df = pd.concat(bench_hist_parts, ignore_index=True)

compare_df = pd.concat([hist_df, bench_hist_df], ignore_index=True)
fig = px.line(compare_df, x="date", y="portfolio_value", color="strategy", title="Stratejiler vs basit benchmark'lar")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
fig.show()


## 13) Latest weights and sleeve summary

Burada iki şeyi görüyoruz:
- son rebalance anında strateji neleri ne kadar almış?
- bunu hangi sleeve'e dağıtmış?


In [15]:
last_weights = weights_df.sort_values("date").groupby("strategy").tail(1).copy()
latest_long = last_weights.melt(id_vars=["date", "strategy", "capital_after_contribution"], var_name="asset", value_name="weight")
latest_long = latest_long[latest_long["asset"].str.startswith("weight_")].copy()
latest_long["asset"] = latest_long["asset"].str.replace("weight_", "", regex=False)
latest_long = latest_long[latest_long["weight"] > 0].copy()

fig = px.bar(
    latest_long.sort_values(["strategy", "weight"], ascending=[True, False]),
    x="asset", y="weight", color="strategy", facet_row="strategy",
    title="Son rebalance anında strateji bazında ağırlıklar"
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

def sleeve_of(asset):
    if asset in ["USDTRY", "EURTRY"]:
        return "FX"
    if asset in ["ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]:
        return "METALS"
    return "BIST"

latest_long["sleeve"] = latest_long["asset"].map(sleeve_of)
sleeve_summary = latest_long.groupby(["strategy", "sleeve"])["weight"].sum().reset_index()
sleeve_pivot = sleeve_summary.pivot(index="strategy", columns="sleeve", values="weight").fillna(0.0)
for c in sleeve_pivot.columns:
    sleeve_pivot[c] = sleeve_pivot[c].map(pct_fmt)
display(sleeve_pivot.reset_index())


sleeve,strategy,BIST,FX,METALS
0,BL,62.83%,20.00%,17.17%
1,Equal,93.24%,2.70%,4.05%
2,HRP,34.00%,59.51%,6.49%
3,HYBRID,48.36%,39.84%,11.81%


## 14) Concentration metrics

Bu bölüm şu soruyu yanıtlar:

**Getiri iyi görünüyor ama gerçekten çeşitlenmiş bir portföy mü?**


In [16]:
concentration_rows = []
for s in STRATEGY_CFG["strategies"]:
    sub = weights_df[weights_df["strategy"] == s].sort_values("date").copy()
    weight_cols = [c for c in sub.columns if c.startswith("weight_")]
    for _, row in sub.iterrows():
        w = row[weight_cols].fillna(0.0).astype(float).values
        w = np.sort(w)[::-1]
        top1 = w[0] if len(w) else np.nan
        top3 = w[:3].sum() if len(w) else np.nan
        eff_n = 1 / np.sum(np.square(w)) if np.sum(np.square(w)) > 0 else np.nan
        concentration_rows.append({"date": row["date"], "strategy": s, "top1_weight": top1, "top3_weight": top3, "effective_n": eff_n})
concentration_df = pd.DataFrame(concentration_rows)

fig = px.line(concentration_df, x="date", y="top1_weight", color="strategy", title="Zaman içinde en büyük tek ağırlık")
fig.show()

fig = px.line(concentration_df, x="date", y="effective_n", color="strategy", title="Zaman içinde efektif varlık sayısı")
fig.show()


## 15) Turnover metrics

Bu bölüm stratejinin ne kadar sık fikir değiştirdiğini gösterir.
Çok turnover = daha düşük güven ve daha fazla sürtünme.


In [17]:
turnover_rows = []
for s in STRATEGY_CFG["strategies"]:
    sub = weights_df[weights_df["strategy"] == s].sort_values("date").copy()
    weight_cols = [c for c in sub.columns if c.startswith("weight_")]
    prev = None
    for _, row in sub.iterrows():
        cur = row[weight_cols].fillna(0.0).astype(float)
        turnover = np.nan if prev is None else np.abs(cur.values - prev.values).sum()
        turnover_rows.append({"date": row["date"], "strategy": s, "turnover": turnover})
        prev = cur
turnover_df = pd.DataFrame(turnover_rows)

fig = px.line(turnover_df.dropna(), x="date", y="turnover", color="strategy", title="Zaman içinde turnover")
fig.show()

avg_turnover = turnover_df.groupby("strategy")["turnover"].mean().reset_index()
fig = px.bar(avg_turnover, x="strategy", y="turnover", title="Ortalama turnover")
fig.show()


## 16) Weight heatmap

Bu ısı haritası zaman içinde ağırlıkların nasıl evrildiğini gösterir.
- hep aynı birkaç satır koyuysa, strateji daralıyor olabilir
- çok kaotik görünüyorsa, strateji fazla oynak davranıyor olabilir


In [18]:
weight_cols = [c for c in weights_df.columns if c.startswith("weight_")]
cumw = weights_df[weight_cols].sum().sort_values(ascending=False).head(15).index.tolist()

heat = weights_df[["date", "strategy"] + cumw].copy()
heat["date"] = heat["date"].dt.strftime("%Y-%m-%d")
heat["row"] = heat["strategy"] + " | " + heat["date"]
heat = heat.set_index("row")[cumw]
heat.columns = [c.replace("weight_", "") for c in heat.columns]

fig = px.imshow(heat.T, aspect="auto", title="Ağırlık evrimi ısı haritası (en çok kullanılan 15 ağırlık sütunu)")
fig.update_xaxes(title="Strateji | Rebalance tarihi")
fig.update_yaxes(title="Varlık")
fig.show()


## 17) BL diagnostics

Bu tablo özellikle BL ve HYBRID içinde hangi view'ların üretildiğini gösterir.
- sürekli tavana vuran view'lar varsa BL kaba olabilir
- hep aynı varlıklar ekstrem görünüyorsa BL tematik olarak dar olabilir


In [19]:
if len(diag_df):
    diag_show = diag_df.copy()
    diag_show["view_pct"] = diag_show["view"].map(pct_fmt)
    display(diag_show.tail(30))


,asset,view,view_source,date,strategy,view_pct
858,PETKM,0.105036,rules,2026-03-02,HYBRID,10.50%
859,PGSUS,-0.150000,rules,2026-03-02,HYBRID,-15.00%
860,REEDR,-0.150000,rules,2026-03-02,HYBRID,-15.00%
861,SASA,-0.150000,rules,2026-03-02,HYBRID,-15.00%
862,SAHOL,-0.050034,rules,2026-03-02,HYBRID,-5.00%
863,SDTTR,0.150000,rules,2026-03-02,HYBRID,15.00%
864,SISE,0.150000,rules,2026-03-02,HYBRID,15.00%
865,SKBNK,0.150000,rules,2026-03-02,HYBRID,15.00%
866,SMRTG,-0.150000,rules,2026-03-02,HYBRID,-15.00%
867,SOKM,0.150000,rules,2026-03-02,HYBRID,15.00%


## 18) What would I trust?

Bu notebook'un en insani kısmı.
Amaç grafik göstermek değil, hangi sonucun daha güven verici göründüğünü söylemek.


In [20]:
best_final = summary_numeric.loc[summary_numeric["ending_value"].idxmax(), "strategy"]
best_dd = summary_numeric.loc[summary_numeric["maxdd"].idxmax(), "strategy"]
best_sharpe = summary_numeric.loc[summary_numeric["sharpe"].idxmax(), "strategy"]

interpretation = f'''
Best pure growth: {best_final}
Best drawdown control: {best_dd}
Best risk-adjusted result: {best_sharpe}

What to trust most right now:
- If one strategy has only a tiny return edge but much worse concentration, distrust it.
- If one strategy is slightly weaker on return but much better on drawdown and turnover, it may be more robust.
- If BL or HYBRID keep showing extreme view behavior, treat them as research tools, not final allocation engines yet.
'''
print(interpretation)



Best pure growth: BL
Best drawdown control: HRP
Best risk-adjusted result: BL

What to trust most right now:
- If one strategy has only a tiny return edge but much worse concentration, distrust it.
- If one strategy is slightly weaker on return but much better on drawdown and turnover, it may be more robust.
- If BL or HYBRID keep showing extreme view behavior, treat them as research tools, not final allocation engines yet.



## 19) Appendix — raw tables

Grafiklerden sonra detay tablo görmek isteyenler için.


In [21]:
out = last_weights[["strategy"] + [c for c in last_weights.columns if c.startswith("weight_")]].copy()
out.columns = ["Strategy"] + [c.replace("weight_", "") for c in out.columns[1:]]
for c in out.columns[1:]:
    out[c] = out[c].map(pct_fmt)
display(out.reset_index(drop=True))

print("Weights by rebalance:")
display(weights_df)

print("Portfolio value path:")
display(hist_df.tail(20))


,Strategy,AEFES,AGHOL,AKBNK,AKFYE,AKSA,AKSEN,ALARK,ARCLK,ASELS,ASTOR,BIMAS,BOBET,CCOLA,CIMSA,CWENE,DOAS,DOHOL,ECILC,EGEEN,EKGYO,ENERY,ENJSA,ENKAI,EREGL,EUPWR,FROTO,GARAN,GESAN,GUBRF,HALKB,HEKTS,ISCTR,ISMEN,KAYSE,KCAER,KCHOL,KLSER,KRDMD,MAVI,MGROS,MIATK,ODAS,OTKAR,OYAKC,PETKM,PGSUS,REEDR,SASA,SAHOL,SDTTR,SISE,SKBNK,SMRTG,SOKM,TAVHL,TCELL,THYAO,TKFEN,TOASO,TSKB,TTKOM,TTRAK,TUPRS,ULKER,VAKBN,VESBE,VESTL,YKBNK,ZOREN,USDTRY,EURTRY,ALTIN_TRY,GUMUS_TRY,PLATIN_TRY
0,HRP,0.31%,0.39%,0.29%,0.72%,0.38%,0.84%,0.53%,0.50%,0.30%,0.33%,0.42%,0.80%,0.45%,0.41%,0.65%,0.57%,0.66%,0.89%,0.77%,0.17%,1.28%,0.57%,0.49%,0.46%,0.20%,0.51%,0.16%,0.31%,0.79%,0.18%,0.35%,0.31%,0.34%,1.71%,0.31%,0.35%,0.77%,0.36%,0.66%,0.35%,0.65%,0.40%,0.39%,0.33%,0.56%,0.44%,0.75%,0.28%,0.31%,0.22%,0.33%,0.65%,0.28%,0.46%,0.34%,0.45%,0.52%,1.32%,0.35%,0.15%,0.38%,0.83%,0.69%,0.38%,0.23%,0.41%,0.34%,0.45%,0.26%,29.76%,29.76%,4.16%,0.80%,1.52%
1,Equal,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%
2,BL,0.00%,0.00%,1.84%,0.00%,0.00%,1.47%,0.00%,0.00%,4.35%,0.00%,5.61%,0.00%,1.64%,0.00%,0.00%,0.00%,9.39%,0.00%,0.00%,0.00%,3.00%,2.22%,4.14%,2.84%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,1.64%,0.00%,0.00%,0.00%,0.00%,0.00%,7.13%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,4.96%,0.00%,1.40%,0.00%,0.35%,0.88%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,10.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,10.00%,10.00%,10.00%,3.80%,3.37%
3,HYBRID,0.16%,0.20%,1.06%,0.36%,0.19%,1.15%,0.27%,0.25%,2.32%,0.17%,3.00%,0.40%,1.04%,0.21%,0.33%,0.29%,5.01%,0.44%,0.39%,0.09%,2.14%,1.39%,2.30%,1.65%,0.10%,0.26%,0.08%,0.16%,0.40%,0.09%,0.18%,0.15%,0.17%,1.67%,0.16%,0.18%,0.39%,0.18%,0.33%,3.72%,0.32%,0.20%,0.20%,0.16%,0.28%,0.22%,0.38%,0.14%,0.16%,2.58%,0.17%,1.02%,0.14%,0.40%,0.61%,0.23%,0.26%,0.66%,0.18%,0.08%,0.19%,0.42%,5.33%,0.19%,0.12%,0.20%,0.17%,0.23%,0.13%,19.92%,19.92%,7.07%,2.30%,2.44%


Weights by rebalance:


,date,strategy,capital_after_contribution,weight_AEFES,weight_AGHOL,weight_AKBNK,weight_AKFYE,weight_AKSA,weight_AKSEN,weight_ALARK,weight_ARCLK,weight_ASELS,weight_ASTOR,weight_BIMAS,weight_BOBET,weight_CCOLA,weight_CIMSA,weight_CWENE,weight_DOAS,weight_DOHOL,weight_ECILC,weight_EGEEN,weight_EKGYO,weight_ENERY,weight_ENJSA,weight_ENKAI,weight_EREGL,weight_EUPWR,weight_FROTO,weight_GARAN,weight_GESAN,weight_GUBRF,weight_HALKB,weight_HEKTS,weight_ISCTR,weight_ISMEN,weight_KAYSE,weight_KCAER,weight_KCHOL,weight_KLSER,weight_KRDMD,weight_MAVI,weight_MGROS,weight_MIATK,weight_ODAS,weight_OTKAR,weight_OYAKC,weight_PETKM,weight_PGSUS,weight_REEDR,weight_SASA,weight_SAHOL,weight_SDTTR,weight_SISE,weight_SKBNK,weight_SMRTG,weight_SOKM,weight_TAVHL,weight_TCELL,weight_THYAO,weight_TKFEN,weight_TOASO,weight_TSKB,weight_TTKOM,weight_TTRAK,weight_TUPRS,weight_ULKER,weight_VAKBN,weight_VESBE,weight_VESTL,weight_YKBNK,weight_ZOREN,weight_USDTRY,weight_EURTRY,weight_ALTIN_TRY,weight_GUMUS_TRY,weight_PLATIN_TRY
0,2025-10-01,Equal,25000.000000,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
1,2025-11-03,Equal,50026.762332,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
2,2025-12-01,Equal,74842.808006,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
3,2026-01-02,Equal,102230.786519,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
4,2026-02-02,Equal,139814.135603,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0

Portfolio value path:


,date,strategy,portfolio_value
492,2026-03-03,HYBRID,164851.426938
493,2026-03-04,HYBRID,164499.050078
494,2026-03-05,HYBRID,165590.516315
495,2026-03-06,HYBRID,164744.189778
496,2026-03-09,HYBRID,163790.032093
497,2026-03-10,HYBRID,167511.747908
498,2026-03-11,HYBRID,167068.975517
499,2026-03-12,HYBRID,167767.127067
500,2026-03-13,HYBRID,166667.702461
501,2026-03-16,HYBRID,165573.545587
